# Переименовать каналы многоканального растра

In [ ]:
import arcpy
from osgeo import gdal
import pathlib

## Пользовательская конфигурация

In [112]:
workdir = pathlib.Path("I:/docs/maxim/prj/gis/learn/FloodBenineSmail/Maps")
rasters_path = workdir / "Rasters"
raster_path = rasters_path / "test.tif"
new_band_names = ["nt1", "t2", "t3"]

## Примечание

Объект arcpy.Raster не может сам по себе нормально изменить название канала, т.к. он почему-то в aux-файле у всех каналов после  первого не меняет элемент Description. Приходится активировать библиотеку gdal, которая доделает всю работу.

В то же время нельзя обойтись только gdal, т.к. если существует aux-файл, то в интерфейсе отображается название канала из нее.

In [113]:
print("Подключаемся к растру ...")
# Получаем доступ к растру через arcPy
rast_obj = arcpy.Raster(str(raster_path))
# Открываем файл как бинарник на редактирование (1 = Update)
ds = gdal.Open(str(raster_path), 1)

Подключаемся к растру ...


In [114]:
if ds is None:
    print("Ошибка: Не могу открыть файл. Возможно, он занят ArcGIS'ом.")
    print("Закрой карту или удали слой перед запуском.")
else:
    print(f"Файл открыт. Каналов: {ds.RasterCount}")
    
    for i, new_name in enumerate(new_band_names, start=1):
        print(f"   Канал '{rast_obj.bandNames[i-1]}' -> присваиваем имя '{new_name}'")
        
        if new_name not in rast_obj.bandNames:            
            rast_obj.renameBand(i, new_name)
        else:
            print(f"   Канал с именем '{new_name}' уже существует, arcPy это не одобряет")

        # Получаем доступ к каналу через gdal
        band = ds.GetRasterBand(i)
        if band:
            # ArcGIS смотрит сюда (Description)
            band.SetDescription(new_name)
            # И сюда (Metadata)
            band.SetMetadataItem("BandName", new_name)
            print(f"Канал {i}: имя записано -> {new_name}")
    
    # Сброс буфера на диск
    band = None
    ds = None
    rast_obj = None
    print("Успех.")

Файл открыт. Каналов: 4
   Канал 'ynt1' -> присваиваем имя 'nt1'
Канал 1: имя записано -> nt1
   Канал 'ynt2' -> присваиваем имя 't2'
Канал 2: имя записано -> t2
   Канал 'ynt3' -> присваиваем имя 't3'
Канал 3: имя записано -> t3
Успех.
